In [ ]:
import pandas as pd

# 1. Isolate orphaned contacts (Contacts missing from your successful ACR export)
linked_contact_ids = set(df_acr['Contact_Source_ID'])
df_orphans = df_contacts[~df_contacts['FinServ__SourceSystemId__c'].isin(linked_contact_ids)].copy()

# 2. Build synthetic composite keys for exact matching
df_orphans['match_key'] = (
    df_orphans['last_name'].str.strip().str.lower() + '_' + 
    df_orphans['Primary_zip'].str[:5].fillna('')
)

# Extract last name from Household Name (assumes 'LastName Household' format)
df_HH['clean_hh_name'] = df_HH['Name'].str.replace('Household', '', case=False).str.strip()
df_HH['match_key'] = (
    df_HH['clean_hh_name'].str.split().str[-1].str.lower() + '_' + 
    df_HH['Primary_zip'].str[:5].fillna('')
)

# 3. Inner merge to generate missing high-confidence ACRs
exact_matches = df_orphans.merge(
    df_HH[['FinServ__SourceSystemId__c', 'match_key', 'Name']], 
    on='match_key', 
    how='inner',
    suffixes=('_Contact', '_HH')
)

# 4. Format for Data Loader
df_auto_acr = exact_matches.rename(columns={
    'FinServ__SourceSystemId__c_Contact': 'FinServ__SourceSystemId__c_Contact',
    'FinServ__SourceSystemId__c_HH': 'FinServ__SourceSystemId__c_HH'
})
df_auto_acr['Roles'] = 'Client'
df_auto_acr['FinServ__PrimaryGroup__c'] = True